# Практика: логирование CNN в MLflow

Этот ноутбук показывает полный цикл для нейросети:
1. Подготовка данных (Digits 8x8)
2. Обучение простой сверточной сети (PyTorch)
3. Логирование метрик и модели в MLflow
4. Пометка версии как `PRD` и загрузка по alias `prd`

In [ ]:
import os
import random
import numpy as np

import mlflow
import mlflow.pytorch
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
# Настройки подключения
os.environ["MLFLOW_TRACKING_URI"] = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5050")

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
print("MLflow URI:", mlflow.get_tracking_uri())

In [ ]:
# Для воспроизводимости
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1) Данные

In [ ]:
digits = load_digits()
X = digits.images.astype(np.float32) / 16.0  # [N, 8, 8]
y = digits.target.astype(np.int64)

X = X[:, None, :, :]  # [N, 1, 8, 8]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

X_train_t = torch.tensor(X_train)
y_train_t = torch.tensor(y_train)
X_test_t = torch.tensor(X_test)
y_test_t = torch.tensor(y_test)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=256, shuffle=False)

print("train:", X_train_t.shape, "test:", X_test_t.shape)

## 2) Модель

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 8x8 -> 4x4
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 4x4 -> 2x2
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 2 * 2, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

## 3) Эксперимент MLflow + обучение + логирование

In [ ]:
experiment_name = "students-cnn-demo-proxy"
artifact_location = "mlflow-artifacts:/"
registered_model_name = "students_cnn_digits"

client = MlflowClient()
exp = mlflow.get_experiment_by_name(experiment_name)
if exp is None:
    exp_id = client.create_experiment(
        name=experiment_name,
        artifact_location=artifact_location
    )
else:
    exp_id = exp.experiment_id

mlflow.set_experiment(experiment_name)

params = {
    "epochs": 8,
    "lr": 0.001,
    "batch_size": 64,
    "optimizer": "Adam",
    "seed": SEED
}

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN(n_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"])

with mlflow.start_run(experiment_id=exp_id):
    mlflow.log_params(params)

    for epoch in range(params["epochs"]):
        model.train()
        running_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        mlflow.log_metric("train_loss", float(epoch_loss), step=epoch)

    model.eval()
    all_logits = []
    all_targets = []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            logits = model(xb).cpu()
            all_logits.append(logits)
            all_targets.append(yb)

    logits = torch.cat(all_logits, dim=0).numpy()
    y_true = torch.cat(all_targets, dim=0).numpy()
    y_pred = logits.argmax(axis=1)

    metrics = {
        "test_accuracy": float(accuracy_score(y_true, y_pred)),
        "test_f1_macro": float(f1_score(y_true, y_pred, average="macro")),
    }
    mlflow.log_metrics(metrics)

    signature = infer_signature(X_test_t[:8].numpy(), logits[:8])

    model_info = mlflow.pytorch.log_model(
        pytorch_model=model,
        artifact_path="model",
        signature=signature,
        input_example=X_test_t[:4].numpy(),
        registered_model_name=registered_model_name,
    )

    new_version = model_info.registered_model_version
    client.set_model_version_tag(registered_model_name, new_version, "env", "PRD")
    client.set_registered_model_alias(registered_model_name, "prd", new_version)

    print("Run ID:", mlflow.active_run().info.run_id)
    print("Registered model version:", new_version)
    print("Alias 'prd' points to version:", new_version)
    print("Metrics:", metrics)

## 4) Загрузка модели по alias `prd`

In [ ]:
loaded_model = mlflow.pytorch.load_model(f"models:/{registered_model_name}@prd")
loaded_model.eval()

with torch.no_grad():
    sample_logits = loaded_model(X_test_t[:3])
    sample_pred = torch.argmax(sample_logits, dim=1).numpy()

print("Sample classes:", sample_pred)